In [1]:
from selenium.common.exceptions import InvalidArgumentException, TimeoutException, WebDriverException
from selenium.webdriver.chrome.service import Service
from selenium import webdriver

from bs4 import BeautifulSoup
import time

In [13]:
def get_soup(path_html):
    with open(path_html, "r", encoding="latin-1") as f:
        soup = BeautifulSoup(f, "html.parser")
    return soup

path_html = 'data/cv.html'
soup = get_soup(path_html)

In [ ]:
artigos_completos = soup.find("div", {"id": "artigos-completos"})
list_artigos = artigos_completos.find_all("div", class_="artigo-completo")

In [ ]:
def get_artigos(soup):
    artigos_completos = soup.find("div", {"id": "artigos-completos"})
    
    
    return list_artigos

list_artigos = get_artigos(soup)
len(list_artigos)

4

In [43]:
for artigo in list_artigos:
    citacoes = artigo.find("span", class_="citacoes")
    print(citacoes)

<span class="citacoes" cvuri="/buscatextual/servletcitacoes?doi=10.1016/j.fsi.2025.110959&amp;issn=10504648&amp;volume=168&amp;issue=&amp;paginaInicial=110959&amp;titulo=Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish&amp;sequencial=1&amp;nomePeriodico=FISH &amp; SHELLFISH IMMUNOLOGY" tooltip="CitaÃ§Ãµes a partir de 1996"></span>
<span class="citacoes" cvuri="/buscatextual/servletcitacoes?doi=10.1111/jfb.70326&amp;issn=00221112&amp;volume=00&amp;issue=&amp;paginaInicial=00&amp;titulo=Impacts of the 2023 Amazon drought in contrasting aquatic ecosystems ( - Scientific Expedition 2023)&amp;sequencial=2&amp;nomePeriodico=JOURNAL OF FISH BIOLOGY" tooltip="CitaÃ§Ãµes a partir de 1996"></span>
<span class="citacoes" cvuri="/buscatextual/servletcitacoes?doi=10.1007/s10646-025-03022-3&amp;issn=09639292&amp;volume=35&amp;issue=&amp;paginaInicial=00&amp;titulo=Combined impacts of warming and methomyl on neurophysiologic

# artigos-completos

In [35]:
from urllib.parse import urlparse, parse_qs, unquote
import httpx

In [44]:
artigo = list_artigos[0]
citado = artigo.find("span", class_="citado")
cvuri = citado['cvuri']
print(cvuri)

/buscatextual/servletcitacoes?doi=10.1016/j.fsi.2025.110959&issn=10504648&volume=168&issue=&paginaInicial=110959&titulo=Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish&sequencial=1&nomePeriodico=FISH & SHELLFISH IMMUNOLOGY


In [47]:
parsed = urlparse(cvuri)
params = parse_qs(parsed.query, keep_blank_values=True)

In [36]:
c_doi = []
s_doi = []
for artigo in list_artigos:
    citacao = artigo.find("span", class_="citado")
    if citacao:
        parsed = urlparse(citacao['cvuri'])
        params = parse_qs(parsed.query, keep_blank_values=True)
        [doi] = params['doi'] 
        if doi == '':
            s_doi.append(params)
            print(doi)
        else:
            c_doi.append(params)
            print(doi)


In [8]:
import httpx
import json
from html import unescape
import re

In [9]:
def limpar_texto(texto: str) -> str:
    if not texto:
        return texto

    # converte entidades HTML, se houver
    texto = unescape(texto)

    # remove tags como <scp>...</scp>
    texto = re.sub(r"<[^>]+>", "", texto)

    # normaliza espaços e quebras de linha
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

In [2]:
doi = '10.1111/jfb.70326'
url = f"https://api.crossref.org/v1/works/{doi}"
r = httpx.get(url)
r.status_code

200

In [11]:
item = r.json()['message']
title = item['title'][0]
title

'Impacts of the 2023 Amazon drought in contrasting aquatic ecosystems (\n                    <scp>INCT ADAPTA</scp>\n                    – Scientific Expedition 2023)'

In [12]:
value = str(title).strip()
value

'Impacts of the 2023 Amazon drought in contrasting aquatic ecosystems (\n                    <scp>INCT ADAPTA</scp>\n                    – Scientific Expedition 2023)'

In [10]:
limpar_texto(title)

'Impacts of the 2023 Amazon drought in contrasting aquatic ecosystems ( INCT ADAPTA – Scientific Expedition 2023)'

In [4]:
with open('data/doi.json', 'w', encoding='utf-8') as f:
    json.dump(item, f, ensure_ascii=False, indent=4)

In [ ]:
error = []
with open("data/artigos/val.jsonl", "w", encoding="utf-8") as f:
    
    for i in c_doi:
        doi = i['doi'][0]
        url = f"https://api.crossref.org/v1/works/{doi}"
        r = httpx.get(url)
        print(r.status_code)
        if r.status_code == 200:
            item = r.json()['message']
            json.dump(item, f)
            f.write("\n")
        else:
            print(f"Error fetching data for DOI: {doi}, status code: {r.status_code}")
            error.append(i)